In [1]:
import subprocess

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

sh("docker --version")
sh("uname -m")

Docker version 29.1.3, build 29.1.3-0ubuntu4.1

aarch64



CompletedProcess(args='uname -m', returncode=0, stdout='aarch64\n', stderr='')

In [2]:
%%writefile generate_dataset.py
# Generates the spam/ham dataset for Assignment 2
import argparse
import random
import pandas as pd

SPAM_TEMPLATES = [
    "WIN a FREE {prize} now! Click here: {url}",
    "Congratulations! You have WON a {prize}. Claim NOW at {url}",
    "URGENT: Your account will be suspended. Verify at {url}",
    "Limited time offer! Get {prize} FREE, click {url} today",
    "You've been selected for a {prize}! Reply YES to claim",
    "Cash prize alert: claim your {prize} before it expires! {url}",
]
HAM_TEMPLATES = [
    "Hey, are we still meeting for {activity} on {day}?",
    "Can you send me the notes from {activity} class?",
    "Don't forget about {activity} this {day}, see you there",
    "Thanks for helping with {activity} yesterday",
    "Running a bit late for {activity}, be there in 10 min",
    "What time does {activity} start on {day}?",
]
PRIZES = ["iPhone", "cash prize", "gift card", "vacation", "laptop"]
URLS = ["bit.ly/xyz123", "tinyurl.com/abc", "win-now.co/claim"]
ACTIVITIES = ["lunch", "the study group", "basketball", "the project meeting"]
DAYS = ["Monday", "Friday", "tomorrow", "the weekend"]


def generate():
    random.seed(42)
    rows = []
    for i in range(1000):
        if random.random() < 0.3:
            t = random.choice(SPAM_TEMPLATES)
            msg = t.format(prize=random.choice(PRIZES), url=random.choice(URLS))
            rows.append((msg, "spam"))
        else:
            t = random.choice(HAM_TEMPLATES)
            msg = t.format(activity=random.choice(ACTIVITIES), day=random.choice(DAYS))
            rows.append((msg, "ham"))
    return pd.DataFrame(rows, columns=["text", "label"])


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default="data/spam_dataset.csv")
    args = parser.parse_args()
    generate().to_csv(args.out, index=False)

Writing generate_dataset.py


In [3]:
import os

os.makedirs("data", exist_ok=True)
subprocess.run(["python3", "generate_dataset.py", "--out", "data/spam_dataset.csv"], check=True)
print("Dataset ready.")

Dataset ready.


In [4]:
%%writefile train_and_export_model.py
# Trains TF-IDF + Naive Bayes on the spam dataset and saves it for the API
import argparse
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/spam_dataset.csv")
    parser.add_argument("--out", default="data/model.joblib")
    args = parser.parse_args()

    df = pd.read_csv(args.data)
    X_train, X_test, y_train, y_test = train_test_split(
        df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"accuracy={accuracy_score(y_test, preds):.4f}")

    joblib.dump(model, args.out)
    print(f"saved model to {args.out}")


if __name__ == "__main__":
    main()

Writing train_and_export_model.py


In [5]:
subprocess.run(["python3", "train_and_export_model.py"], check=True)
print(os.path.getsize("data/model.joblib"), "bytes")

accuracy=1.0000
saved model to data/model.joblib
5889 bytes


In [6]:
%%writefile spam_app.py
# Spam detection API
import os
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH = os.environ.get("MODEL_PATH", "data/model.joblib")

app = FastAPI(title="Spam Detection API")
_model = None


@app.on_event("startup")
def load_model():
    global _model
    _model = joblib.load(MODEL_PATH)
    print(f"loaded model from {MODEL_PATH}")


class PredictRequest(BaseModel):
    text: str


@app.get("/healthz")
def healthz():
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"status": "ok"}


@app.post("/predict")
def predict(request: PredictRequest):
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    return {"label": _model.predict([request.text]).tolist()[0]}

Writing spam_app.py


In [7]:
import subprocess, time, requests, os

env = os.environ.copy()
env.update({"MODEL_PATH": "data/model.joblib"})

proc = subprocess.Popen(
    ["uvicorn", "spam_app:app", "--host", "0.0.0.0", "--port", "28080"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(3)

In [8]:
health = requests.get("http://localhost:28080/healthz")
print("Health check:", health.status_code, health.json())

for text in ["WIN a FREE iPhone now! Click here: bit.ly/xyz123",
             "Hey, are we still meeting for lunch on Friday?"]:
    pred = requests.post("http://localhost:28080/predict", json={"text": text}).json()
    print(text, "->", pred)

Health check: 200 {'status': 'ok'}
WIN a FREE iPhone now! Click here: bit.ly/xyz123 -> {'label': 'spam'}
Hey, are we still meeting for lunch on Friday? -> {'label': 'ham'}


In [9]:
proc.terminate()
proc.wait(timeout=5)
print("Local test server stopped.")

Local test server stopped.


In [10]:
%%writefile requirements-predictor.txt
fastapi
uvicorn[standard]
scikit-learn
joblib
pydantic

Writing requirements-predictor.txt


In [11]:
%%writefile Dockerfile.naive
FROM python:3.11

WORKDIR /app

COPY . .
RUN pip install -r requirements-predictor.txt

ENV MODEL_PATH=/app/data/model.joblib

EXPOSE 8080
CMD ["uvicorn", "spam_app:app", "--host", "0.0.0.0", "--port", "8080"]

Writing Dockerfile.naive


In [12]:
sh("docker run -d --name spam-naive -p 8080:8080 spam-api:naive")
time.sleep(5)
sh("docker ps")

e4a19660b289ed6d44b77090996a5716303e16dc0e0e7e79ea34c56143ae74c0

CONTAINER ID   IMAGE            COMMAND                  CREATED         STATUS         PORTS                                         NAMES
e4a19660b289   spam-api:naive   "uvicorn spam_app:ap…"   5 seconds ago   Up 5 seconds   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp   spam-naive



CompletedProcess(args='docker ps', returncode=0, stdout='CONTAINER ID   IMAGE            COMMAND                  CREATED         STATUS         PORTS                                         NAMES\ne4a19660b289   spam-api:naive   "uvicorn spam_app:ap…"   5 seconds ago   Up 5 seconds   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp   spam-naive\n', stderr='')

In [13]:
health = requests.get("http://localhost:8080/healthz")
print("Health check:", health.status_code, health.json())

for text in ["WIN a FREE iPhone now! Click here: bit.ly/xyz123",
             "Hey, are we still meeting for lunch on Friday?"]:
    pred = requests.post("http://localhost:8080/predict", json={"text": text}).json()
    print(text, "->", pred)

Health check: 200 {'status': 'ok'}
WIN a FREE iPhone now! Click here: bit.ly/xyz123 -> {'label': 'spam'}
Hey, are we still meeting for lunch on Friday? -> {'label': 'ham'}


In [16]:
sh("docker images spam-api");
sh("docker rm -f spam-naive");

IMAGE            ID             DISK USAGE   CONTENT SIZE   EXTRA
spam-api:naive   1e4cedd29aa7       2.19GB          565MB        




In [17]:
%%writefile Dockerfile.multistage
# Stage 1: install dependencies
FROM python:3.11 AS builder

WORKDIR /app

RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

COPY requirements-predictor.txt .
RUN pip install --no-cache-dir -U pip
RUN pip install --no-cache-dir -r requirements-predictor.txt

# Stage 2: runtime
FROM python:3.11-slim AS runner

WORKDIR /app

COPY --from=builder /opt/venv /opt/venv
COPY spam_app.py .
COPY data/model.joblib ./model.joblib
ENV MODEL_PATH=/app/model.joblib

ENV PATH="/opt/venv/bin:$PATH"

EXPOSE 8080
CMD ["uvicorn", "spam_app:app", "--host", "0.0.0.0", "--port", "8080"]

Writing Dockerfile.multistage


In [20]:
health = requests.get("http://localhost:8080/healthz")
print("Health check:", health.status_code, health.json())

for text in ["WIN a FREE iPhone now! Click here: bit.ly/xyz123",
             "Hey, are we still meeting for lunch on Friday?"]:
    pred = requests.post("http://localhost:8080/predict", json={"text": text}).json()
    print(text, "->", pred)

Health check: 200 {'status': 'ok'}
WIN a FREE iPhone now! Click here: bit.ly/xyz123 -> {'label': 'spam'}
Hey, are we still meeting for lunch on Friday? -> {'label': 'ham'}


In [21]:
sh("docker images spam-api")

naive_mb = 2190
multi_mb = 689
print(f"disk usage reduction: {(naive_mb - multi_mb) / naive_mb * 100:.1f}%")

naive_content_mb = 565
multi_content_mb = 148
print(f"content size reduction: {(naive_content_mb - multi_content_mb) / naive_content_mb * 100:.1f}%")

IMAGE                 ID             DISK USAGE   CONTENT SIZE   EXTRA
spam-api:multistage   b5f47dd74806        689MB          148MB   U    
spam-api:naive        1e4cedd29aa7       2.19GB          565MB        

disk usage reduction: 68.5%
content size reduction: 73.8%


In [22]:
sh("docker history spam-api:naive --format 'table {{.Size}}\t{{.CreatedBy}}'")
sh("docker history spam-api:multistage --format 'table {{.Size}}\t{{.CreatedBy}}'");

SIZE      CREATED BY
0B        CMD ["uvicorn" "spam_app:app" "--host" "0.0.…
0B        EXPOSE [8080/tcp]
0B        ENV MODEL_PATH=/app/data/model.joblib
427MB     RUN /bin/sh -c pip install -r requirements-p…
131kB     COPY . . # buildkit
8.19kB    WORKDIR /app
0B        CMD ["python3"]
16.4kB    RUN /bin/sh -c set -eux;  for src in idle3 p…
73.4MB    RUN /bin/sh -c set -eux;   wget -O python.ta…
0B        ENV PYTHON_SHA256=91bcdebfdde239a003ae93738a…
0B        ENV PYTHON_VERSION=3.11.16
0B        ENV GPG_KEY=A035C8C19219BA821ECEA86B64E628F8…
20.5MB    RUN /bin/sh -c set -eux;  apt-get update;  a…
0B        ENV LANG=C.UTF-8
0B        ENV PATH=/usr/local/bin:/usr/local/sbin:/usr…
679MB     RUN /bin/sh -c set -ex;  apt-get update;  ap…
208MB     RUN /bin/sh -c set -eux;  apt-get update;  a…
63.7MB    RUN /bin/sh -c set -eux;  apt-get update;  a…
156MB     # debian.sh --arch 'arm64' out/ 'trixie' '@1…

SIZE      CREATED BY
0B        CMD ["uvicorn" "spam_app:app" "--host" "0.0.…
0B        

In [23]:
for tag in ["naive", "multistage"]:
    print(f"----- spam-api:{tag} -----")
    sh(f"docker run --rm spam-api:{tag} ls -a /app")
    sh(f"docker run --rm spam-api:{tag} sh -c 'which gcc || echo no gcc'")
    sh(f"docker run --rm spam-api:{tag} sh -c 'du -sh /root/.cache/pip 2>/dev/null || echo no pip cache'")
sh("docker rm -f spam-multi");

----- spam-api:naive -----
.
..
.ipynb_checkpoints
Dockerfile.naive
__pycache__
assignment2.ipynb
data
generate_dataset.py
requirements-predictor.txt
spam_app.py
train_and_export_model.py

/usr/bin/gcc

71M	/root/.cache/pip

----- spam-api:multistage -----
.
..
model.joblib
spam_app.py

no gcc

no pip cache

spam-multi

